In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import matplotlib.pyplot as plt
import time
import os
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset


cuda


In [11]:
output_dir = "problem_1_RNN_results"
os.makedirs(output_dir, exist_ok=True)
training_start = time.time()
epoch_start = time.time()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


!rm -f Problem_1_sequence.txt

!wget -L "https://raw.githubusercontent.com/David-Ojo/UNCC-ECGR-4106/main/Assignment%204/Problem_1_sequence.txt" -O Problem_1_sequence.txt
FILE_NAME = "Problem_1_sequence"

with open("Problem_1_sequence.txt", "r") as f:
  text = f.read()

chars = sorted(list(set(text)))
#This line creates a dictionary that maps each character to a unique index (integer)."
ix_to_char = {i: ch for i, ch in enumerate(chars)}
#Similar to the previous line, but in reverse. This line creates a dictionary that maps each unique index (integer) back to its corresponding character.
char_to_ix = {ch: i for i, ch in enumerate(chars)}
chars = sorted(list(set(text)))



cuda
--2026-07-09 01:58:56--  https://raw.githubusercontent.com/David-Ojo/UNCC-ECGR-4106/main/Assignment%204/Problem_1_sequence.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2393 (2.3K) [text/plain]
Saving to: ‘Problem_1_sequence.txt’

Problem_1_sequence. 100%[===================>]   2.34K  --.-KB/s    in 0s      

2026-07-09 01:58:56 (49.3 MB/s) - ‘Problem_1_sequence.txt’ saved [2393/2393]



In [12]:
# Preparing the dataset
max_length = 10  # Maximum length of input sequences
X = []
y = []
for i in range(len(text) - max_length):
    sequence = text[i:i + max_length]
    label = text[i + max_length]
    X.append([char_to_ix[char] for char in sequence])
    y.append(char_to_ix[label])

X = np.array(X)
y = np.array(y)

# Splitting the dataset into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Converting data to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

X_train = torch.tensor(X_train, dtype=torch.long).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_val = torch.tensor(X_val, dtype=torch.long).to(device)
y_val = torch.tensor(y_val, dtype=torch.long).to(device)

/tmp/ipykernel_617/2614088772.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.long).to(device)
/tmp/ipykernel_617/2614088772.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long).to(device)
/tmp/ipykernel_617/2614088772.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val = torch.tensor(X_val, dtype=torch.long).to(device)
/tmp/ipykernel_617/2614088772.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTe

In [21]:
# Defining the RNN model
class CharRNNModel(nn.Module):
    def __init__(self, model_type, vocab_size, hidden_size, num_layers=1):
        super().__init__()

        self.model_type = model_type
        self.embedding = nn.Embedding(vocab_size, hidden_size)

        if model_type == "rnn":
            self.rnn = nn.RNN(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        elif model_type == "gru":
            self.rnn = nn.GRU(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        elif model_type == "lstm":
            self.rnn = nn.LSTM(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out
def train_rnn_model(model_type, seq_len, hidden_size=24, num_layers=1, epochs=100):
    X, y = [], []

    for i in range(len(text) - seq_len):
        sequence = text[i:i + seq_len]
        label = text[i + seq_len]
        X.append([char_to_ix[c] for c in sequence])
        y.append(char_to_ix[label])

    X = np.array(X)
    y = np.array(y)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    X_train = torch.tensor(X_train, dtype=torch.long)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_val = torch.tensor(X_val, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(X_val, y_val),
        batch_size=64,
        shuffle=False
    )

    model = CharRNNModel(
        model_type,
        len(chars),
        hidden_size,
        num_layers
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    train_losses = []
    val_losses = []
    val_accs = []
    epoch_times = []

    best_val_loss = float("inf")
    best_model_state = None
    patience = 10
    patience_counter = 0

    start_time = time.time()
    for epoch in range(epochs):
        epoch_start = time.time()

        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                val_output = model(X_batch)
                loss = criterion(val_output, y_batch)
                val_loss += loss.item()

                _, predicted = torch.max(val_output, 1)
                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss /= len(val_loader)
        val_accuracy = correct / total

        val_losses.append(val_loss)
        val_accs.append(val_accuracy)

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)

        print(
            f"{model_type.upper()} seq={seq_len} "
            f"Epoch {epoch+1}: "
            f"loss={train_loss:.4f}, "
            f"val_loss={val_loss:.4f}, "
            f"val_acc={val_accuracy:.4f}, "
            f"time={epoch_time:.2f}s"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    total_time = time.time() - start_time

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    model_size = param_size / 1024**2

    return {
        "model": model_type,
        "seq_len": seq_len,
        "hidden_size": hidden_size,
        "layers": num_layers,
        "train_loss": train_losses,
        "val_loss": val_losses,
        "val_acc": val_accs,
        "time": total_time,
        "avg_epoch_time": np.mean(epoch_times),
        "trained_model": model,
        "params": total_params,
        "model_size_mb": model_size
    }

In [22]:

rnn_results = []

for model_type in ["rnn", "gru", "lstm"]:
    for seq_len in [10, 20, 30]:
        rnn_results.append(
            train_rnn_model(
                model_type,
                seq_len,
                hidden_size=24,
                num_layers=1,
                epochs=100
            )
        )
def save_result_plots(result):
    model_name = result["model"]
    seq_len = result["seq_len"]

    plt.figure(figsize=(8, 5))
    plt.plot(result["train_loss"], label="Training Loss")
    plt.plot(result["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name.upper()} Loss Curves | Sequence Length {seq_len}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{model_name}_seq{seq_len}_loss.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(result["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name.upper()} Validation Accuracy | Sequence Length {seq_len}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{model_name}_seq{seq_len}_accuracy.png"))
    plt.close()
for r in rnn_results:
    save_result_plots(r)

    summary = []

for r in rnn_results:
    summary.append({
        "model": r["model"],
        "seq_len": r["seq_len"],
        "hidden_size": r["hidden_size"],
        "layers": r["layers"],
        "final_train_loss": r["train_loss"][-1],
        "final_val_loss": r["val_loss"][-1],
        "final_val_acc": r["val_acc"][-1],
        "time_sec": r["time"],
        "avg_epoch_time": r["avg_epoch_time"],
        "trainable_params": r["params"],
        "model_size_MB": r["model_size_mb"]
    })

df_rnn = pd.DataFrame(summary)
df_rnn.to_csv(os.path.join(output_dir, "rnn_gru_lstm_comparison.csv"), index=False)

print(df_rnn)

RNN seq=10 Epoch 1: loss=3.2776, val_loss=2.9759, val_acc=0.1258, time=0.28s
RNN seq=10 Epoch 2: loss=2.8414, val_loss=2.7910, val_acc=0.2369, time=0.10s
RNN seq=10 Epoch 3: loss=2.6279, val_loss=2.6236, val_acc=0.2725, time=0.10s
RNN seq=10 Epoch 4: loss=2.4740, val_loss=2.5043, val_acc=0.3061, time=0.10s
RNN seq=10 Epoch 5: loss=2.3509, val_loss=2.4273, val_acc=0.3291, time=0.10s
RNN seq=10 Epoch 6: loss=2.2518, val_loss=2.3571, val_acc=0.3564, time=0.09s
RNN seq=10 Epoch 7: loss=2.1590, val_loss=2.3040, val_acc=0.3732, time=0.10s
RNN seq=10 Epoch 8: loss=2.0805, val_loss=2.2648, val_acc=0.3753, time=0.10s
RNN seq=10 Epoch 9: loss=2.0151, val_loss=2.2470, val_acc=0.3774, time=0.09s
RNN seq=10 Epoch 10: loss=1.9487, val_loss=2.2063, val_acc=0.4025, time=0.10s
RNN seq=10 Epoch 11: loss=1.8957, val_loss=2.2191, val_acc=0.3899, time=0.09s
RNN seq=10 Epoch 12: loss=1.8597, val_loss=2.2066, val_acc=0.3857, time=0.09s
RNN seq=10 Epoch 13: loss=1.8083, val_loss=2.1748, val_acc=0.4130, time=0